
# ✅ Core Vision Perfect V1 — 03 · Test the System & Trained Model

Sanity-checks the full retrieval stack on Colab: loads the engine, runs
Vietnamese queries end-to-end (KIS / TRAKE / AVS), measures latency, writes a
sample submission CSV, validates + packages it Codabench-style, optionally
scores it against a local ground truth (official formulas) and dry-runs the
automatic track. The Streamlit UI can be served via a tunnel at the end.

Prerequisite: notebook 01 (and optionally 02 for the fine-tuned tower).


In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  1 · PARAMS — the ONLY cell you may need to edit                 ║
# ╚══════════════════════════════════════════════════════════════════╝
DRIVE_PROJECT_DIR = "AIC2025"        # MyDrive/<this>/{data, artifacts}
REPO_URL  = "https://github.com/ledinhminhquan/Core-Vision_Perfect_V1.git"
REPO_REF  = "main"

# Which dense encoders to build indexes for (order = ensemble order).
#   "siglip2"          multilingual default (needed for training too)
#   "openclip"         English lane (DFN5B ViT-H/14-378) — strongest with translation
#   "qwen_embed"       optional HEAVY lane (Qwen embedding tower — strong, slow)
#   "provided_clip32"  organiser features — instant, no GPU (L-batches only);
#                      auto-added in the catalog cell when clip-features-32 exists
EMBED_MODELS = ["siglip2", "openclip"]

# Copy keyframes from Drive → local disk before embedding (much faster I/O).
COPY_KEYFRAMES_LOCAL = True

# Aux indexes to build (each is resumable; captions are the slowest).
RUN_OCR, RUN_ASR, RUN_CAPTIONS = True, True, True
CAPTION_STRIDE = 4                   # caption mỗi keyframe thứ 4 (round-16: đủ dày
#   cho kênh recall BM25 mà nhanh gấp đôi stride 2. NÂNG stride luôn an toàn với
#   resume: video đã caption ở stride nhỏ hơn vẫn được tính là XONG ở stride lớn
#   hơn. Mọi phiên chạy song song PHẢI dùng CÙNG một stride.

# K-batch shot detection: install TransNetV2 (the winning-team detector) for
# keyframe self-extraction. Installed --no-deps (Colab torch is never touched);
# without it extraction falls back to PySceneDetect automatically. (nb01 only)
INSTALL_TRANSNETV2 = True

# Force-rebuild toggles — mặc định False = resume/skip khi artifact đã có.
FORCE_CATALOG    = False             # rebuild the catalog parquet
FORCE_EMBED      = False             # re-embed every keyframe
FORCE_INDEX      = False             # rebuild the FAISS indexes
FORCE_AUX        = False             # redo OCR/ASR/captions from scratch
FORCE_TEXT_INDEX = False             # rebuild the persisted BM25 text index

import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ["HF_HUB_DISABLE_TELEMETRY"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
print("params ok")

In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  2 · Mount Drive + folder layout + preflight write test          ║
# ╚══════════════════════════════════════════════════════════════════╝
import os, shutil, subprocess, time
from pathlib import Path

from google.colab import drive

_MP = "/content/drive"

def _drive_alive() -> bool:
    try:
        return Path(_MP, "MyDrive").exists()
    except OSError:
        return False

def _ensure_drive():
    """Mount / HỒI SINH Drive FUSE — dùng ở mọi cell dài hơi phía sau.

    Round-19 (live run 8): daemon DriveFS chết để lại mountpoint 'bẩn' →
    drive.mount kêu 'Mountpoint must not already contain files' và cả
    force_remount cũng bó tay. Trình tự cứu đúng: (1) fusermount -uz gỡ
    mount chết; (2) CHỈ khi chắc chắn không còn mount (os.path.ismount ==
    False — lúc này các entry trong mountpoint là RÁC LOCAL trên đĩa VM,
    không phải Drive thật) mới dọn sạch chúng; (3) mount lại.
    """
    for _try in range(4):
        if _drive_alive():
            return
        if _try:
            print(f"⚠ Drive FUSE chưa sống — hồi sinh (lần {_try}/3) ...")
        try:
            if os.path.ismount(_MP):
                subprocess.run(["fusermount", "-uz", _MP], capture_output=True)
                time.sleep(2)
            if os.path.isdir(_MP) and not os.path.ismount(_MP):
                for _c in os.listdir(_MP):     # rác local — KHÔNG phải Drive
                    _p = os.path.join(_MP, _c)
                    shutil.rmtree(_p, ignore_errors=True) if os.path.isdir(_p) \
                        else os.unlink(_p)
            drive.mount(_MP, force_remount=bool(_try))
        except Exception as _e:  # noqa: BLE001 — thử tiếp vòng sau
            print("   mount lỗi:", _e)
            time.sleep(5)
    if not _drive_alive():
        raise RuntimeError(
            "Không mount được Google Drive sau 4 lần thử — Runtime ▸ "
            "Disconnect and delete runtime rồi Run all lại (tiến độ đã lưu "
            "trên Drive còn nguyên).")

_ensure_drive()
assert Path("/content/drive/MyDrive").exists(), "Drive mount failed — rerun this cell"

PROJECT   = Path("/content/drive/MyDrive") / DRIVE_PROJECT_DIR
DATA_DIR  = PROJECT / "data"           # organiser dataset (merged packages)
ARTIFACTS = PROJECT / "artifacts"      # everything we build → survives disconnects
for p in (DATA_DIR, ARTIFACTS):
    p.mkdir(parents=True, exist_ok=True)

# PREFLIGHT (v12): Drive PHẢI ghi/đọc được — quota đầy hay mất quyền thì
# dừng NGAY tại đây thay vì hỏng giữa chừng sau 2 giờ chạy.
_probe = ARTIFACTS / f"_write_test_{int(time.time())}.tmp"
try:
    _probe.write_text("ok", encoding="utf-8")
    assert _probe.read_text(encoding="utf-8") == "ok"
    _probe.unlink()
    print("✅ Drive write test: OK")
except Exception as e:
    raise RuntimeError(
        f"❌ Không ghi được vào Drive ({ARTIFACTS}): {e!r}\n"
        "Kiểm tra dung lượng (quota) Google Drive và quyền truy cập thư mục, "
        "rồi chạy lại ô này."
    ) from e

# DATA-PRESENCE GATE (round-20, live run 9): trên VM mới, DriveFS có thể liệt
# kê data/ ra RỖNG suốt vài phút đầu (metadata sync lười) — mkdir exist_ok ở
# trên còn CHE mất triệu chứng, để cell 7 chết khó hiểu với "Keyframes folder
# not found". Poll tới 3 phút (mỗi listdir là một cú hích ép DriveFS fetch);
# hết kiên nhẫn thì dừng TO với chẩn đoán rõ ràng.
_t0 = time.time()
_data_ok = False
while time.time() - _t0 < 180:
    try:
        if any(DATA_DIR.iterdir()):
            _data_ok = True
            break
    except OSError:
        pass
    print(f"⏳ data/ đang rỗng — đợi DriveFS sync metadata ({int(time.time() - _t0)}s) ...")
    time.sleep(10)
if not _data_ok:
    raise RuntimeError(
        "data/ trên Drive vẫn RỖNG sau 3 phút chờ. Ba nguyên nhân thường gặp:\n"
        "  1) Phiên Colab đăng nhập NHẦM tài khoản Google (kiểm tra avatar góc "
        f"phải trên) — phải là tài khoản có MyDrive/{DRIVE_PROJECT_DIR}/data;\n"
        "  2) DriveFS sync quá chậm — Runtime ▸ Disconnect and delete runtime "
        "rồi Run all lại trên máy mới;\n"
        "  3) Lần chạy đầu tiên mà chưa upload dữ liệu — ném các zip của BTC "
        f"vào MyDrive/{DRIVE_PROJECT_DIR}/data trước (docs/DRIVE_SETUP.md).\n"
        "KHÔNG có gì bị mất — dữ liệu vẫn nằm nguyên trên Drive của tài khoản đúng.")
print(f"✅ data/ nhìn thấy dữ liệu sau {int(time.time() - _t0)}s")

# HF + pip caches on Drive → models/wheels download once, not per session.
os.environ["HF_HOME"] = str(ARTIFACTS / "hf_cache")
os.environ["PIP_CACHE_DIR"] = str(ARTIFACTS / "pip_cache")
for _d in (os.environ["HF_HOME"], os.environ["PIP_CACHE_DIR"]):
    Path(_d).mkdir(parents=True, exist_ok=True)

import shutil
free_gb = shutil.disk_usage(str(PROJECT)).free / 1e9
print(f"Project: {PROJECT}")
print(f"Drive free space: {free_gb:.0f} GB")
if free_gb < 20:
    print("⚠ Less than 20 GB free on Drive — embeddings/checkpoints may not fit!")

In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  3 · Get repo + install dependencies (v12 discipline)            ║
# ╚══════════════════════════════════════════════════════════════════╝
# Quy tắc (học từ notebook Toxicity v12):
#   * check version qua importlib.metadata — KHÔNG import package trước khi
#     nâng cấp (import sớm sẽ ghim version cũ vào sys.modules);
#   * KHÔNG BAO GIỜ đụng torch/torchvision/torchaudio của Colab;
#   * chỉ cài đúng những gói thiếu/sai version (--prefer-binary);
#   * sau khi cài: `pip check` + micro-fix (tối đa 2 vòng, không crash),
#     rồi purge sys.modules TRƯỚC khi import cvp.
FORCE_REINSTALL_DEPS = False

import re, subprocess, sys
from pathlib import Path

REPO_DIR = Path("/content/Core-Vision_Perfect_V1")

def _run(cmd, show=None, **kw):
    # `show` masks credentials in the echoed command — a PAT-carrying clone
    # URL must NEVER be printed into the saved notebook output.
    print("$", " ".join(map(str, show or cmd)))
    return subprocess.run([str(c) for c in cmd], check=False, **kw).returncode

def _pip(args):
    return _run([sys.executable, "-m", "pip", *args])

# Private repo? Add a fine-grained PAT as Colab secret "GITHUB_TOKEN"
# (Contents: Read-only on this repo) and ENABLE its notebook-access toggle.
clone_url, _tok = REPO_URL, None
try:
    from google.colab import userdata
    _tok = userdata.get("GITHUB_TOKEN")
except Exception as _e:
    print(f"⚠ KHÔNG đọc được secret GITHUB_TOKEN ({type(_e).__name__}) — repo "
          "private sẽ KHÔNG clone được. Kiểm tra: 🔑 panel có secret tên đúng "
          "y hệt GITHUB_TOKEN và công tắc 'Notebook access' đã BẬT chưa?")
if _tok and clone_url.startswith("https://github.com/"):
    clone_url = clone_url.replace("https://", f"https://{_tok}@")
    print(f"GITHUB_TOKEN: loaded ({len(_tok)} chars, {_tok[:11]}…)")
elif not _tok:
    print("⚠ GITHUB_TOKEN trống/vắng mặt — thử clone KHÔNG xác thực "
          "(chắc chắn fail nếu repo private).")

if REPO_DIR.exists():
    _run(["git", "-C", REPO_DIR, "fetch", "--all", "-q"])
    _run(["git", "-C", REPO_DIR, "checkout", REPO_REF, "-q"])
    _run(["git", "-C", REPO_DIR, "pull", "-q"])
else:
    rc = _run(["git", "clone", "--branch", REPO_REF, clone_url, REPO_DIR],
              show=["git", "clone", "--branch", REPO_REF, REPO_URL, REPO_DIR])
    if rc != 0:  # private repo / no network → fall back to a Drive copy
        print("⚠ Clone THẤT BẠI. Nguyên nhân thường gặp, theo thứ tự:\n"
              "  1) Secret GITHUB_TOKEN sai tên / chưa bật Notebook access "
              "(xem cảnh báo phía trên);\n"
              "  2) PAT sai/hết hạn/thiếu quyền — cần fine-grained PAT với "
              "Contents: Read-only cấp cho ĐÚNG repo này;\n"
              "  3) Mạng Colab trục trặc tạm thời — chạy lại cell.")
        drive_copy = Path("/content/drive/MyDrive") / DRIVE_PROJECT_DIR / "Core-Vision_Perfect_V1"
        assert drive_copy.exists(), (
            "Clone failed and no Drive copy found. Either make the GitHub repo "
            f"reachable or upload the repo folder to {drive_copy}"
        )
        import shutil as _sh
        _sh.copytree(drive_copy, REPO_DIR)
        print("Using repo copy from Drive")

try:
    from packaging.requirements import Requirement
except ImportError:
    _pip(["install", "-q", "packaging"])
    from packaging.requirements import Requirement
from importlib.metadata import PackageNotFoundError
from importlib.metadata import version as _meta_version

# Parse requirements-colab.txt; strip any torch* line (Colab rule #1: the
# preinstalled torch/torchvision/torchaudio build must never be touched).
reqs = []
for _line in (REPO_DIR / "requirements-colab.txt").read_text(encoding="utf-8").splitlines():
    _line = _line.split("#", 1)[0].strip()
    if not _line:
        continue
    try:
        _r = Requirement(_line)
    except Exception:
        print("⚠ bỏ qua requirement không parse được:", _line)
        continue
    if _r.name.lower().replace("-", "_").startswith("torch"):
        print("skip (never touch Colab torch):", _line)
        continue
    reqs.append(_r)

def _satisfied(r):
    """Installed + in range — via importlib.metadata, WITHOUT importing it."""
    try:
        v = _meta_version(r.name)
    except PackageNotFoundError:
        return False
    return (not r.specifier) or r.specifier.contains(v, prereleases=True)

missing = [r for r in reqs if FORCE_REINSTALL_DEPS or not _satisfied(r)]
did_install = bool(missing)
if missing:
    print(f"installing {len(missing)} package(s):", ", ".join(r.name for r in missing))
    _pip(["install", "-q", "--prefer-binary", *[str(r) for r in missing]])
else:
    print("dependencies satisfied — no pip install needed")

_pip(["install", "-q", "-e", str(REPO_DIR), "--no-deps"])

# faiss: gpu wheel with cpu fallback (metadata check — no import)
def _installed(*names):
    for n in names:
        try:
            _meta_version(n)
            return n
        except PackageNotFoundError:
            pass
    return None

if _installed("faiss-gpu-cu12", "faiss-gpu", "faiss-cpu", "faiss") is None:
    if _pip(["install", "-q", "faiss-gpu-cu12"]) != 0:
        _pip(["install", "-q", "faiss-cpu"])
    did_install = True

# `pip check` + micro-fixes for known conflicts (max 2 rounds, then warn)
def _pip_check():
    r = subprocess.run([sys.executable, "-m", "pip", "check"],
                       capture_output=True, text=True)
    return r.returncode, ((r.stdout or "") + "\n" + (r.stderr or "")).strip()

if did_install:
    rc, out = _pip_check()
    for _round in (1, 2):
        if rc == 0:
            break
        # pip's two REAL formats (round-3 fix L-R3-8 — the old regex missed the
        # version-conflict wording so that repair branch never ran):
        #   "pkgA 1.0 requires pkgB, which is not installed."
        #   "pkgA 1.0 has requirement pkgB<2,>=1, but you have pkgB 3.0."
        _specs = sorted({
            m.strip()
            for m in re.findall(
                r"(?:requires|has requirement) (.+?), (?:but you have|which is not installed)", out)
            if not m.strip().lower().startswith("torch")
        })
        if not _specs:
            break
        print(f"pip check micro-fix (round {_round}):", ", ".join(_specs))
        _pip(["install", "-q", "--prefer-binary", *_specs])
        rc, out = _pip_check()
    print("pip check: OK" if rc == 0 else f"⚠ pip check còn cảnh báo (không chặn):\n{out}")

# Purge stale sys.modules of upgraded packages BEFORE importing cvp (v12).
# ONLY the packages actually (re)installed THIS run (round-11): purging every
# requirement dropped numpy/pandas from sys.modules while torch still held
# references to the old modules — the "NumPy module was reloaded" warning.
if did_install:
    _ALIAS = {"pillow": "pil", "pyyaml": "yaml", "opencv_python_headless": "cv2",
              "open_clip_torch": "open_clip", "scikit_learn": "sklearn"}
    _roots = {r.name.lower().replace("-", "_") for r in missing} | {"cvp", "faiss"}
    _roots |= {_ALIAS[n] for n in _roots & set(_ALIAS)}
    _purged = [m for m in list(sys.modules)
               if m.split(".", 1)[0].lower().replace("-", "_") in _roots]
    for _m in _purged:
        sys.modules.pop(_m, None)
    if _purged:
        print(f"purged {len(_purged)} stale sys.modules entries")

    # Sanity (round-11): the HF stack must import cleanly in a FRESH
    # interpreter — a broken hub/accelerate pairing must surface HERE with an
    # actionable message, not 5 cells later as a cryptic circular import.
    _rc = _run([sys.executable, "-c", "import transformers, accelerate"])
    if _rc != 0:
        print("⚠ transformers/accelerate KHÔNG import được — thường do phiên cài "
              "này đã hạ cấp huggingface-hub dưới mức accelerate cần. Cách sửa "
              "sạch nhất: Runtime ▸ Disconnect and delete runtime, rồi Run all "
              "lại từ đầu (mọi tiến độ đã nằm trên Drive, không mất gì).")

if str(REPO_DIR / "src") not in sys.path:
    sys.path.insert(0, str(REPO_DIR / "src"))
import cvp
print("cvp", cvp.__version__, "ready")

In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  4 · Point cvp at the data + GPU setup (TF32 / SDPA / bf16)      ║
# ╚══════════════════════════════════════════════════════════════════╝
import os, torch

os.environ["CVP_PATHS__DATA_ROOT"]      = str(DATA_DIR)
os.environ["CVP_PATHS__ARTIFACTS_ROOT"] = str(ARTIFACTS)
os.environ["CVP_SETTINGS"] = str(REPO_DIR / "configs" / "settings.yaml")

print("torch", torch.__version__, "| CUDA build", torch.version.cuda)
print("GPU available:", torch.cuda.is_available())
GPU_NAME, VRAM_GB, USE_BF16 = "cpu", 0.0, False
if torch.cuda.is_available():
    GPU_NAME = torch.cuda.get_device_name(0)
    VRAM_GB = torch.cuda.get_device_properties(0).total_memory / 1e9
    USE_BF16 = torch.cuda.is_bf16_supported()
    # TF32 fast paths (new API with old fallback)
    try:
        torch.backends.cuda.matmul.fp32_precision = "tf32"
        torch.backends.cudnn.conv.fp32_precision = "tf32"
    except Exception:
        torch.backends.cuda.matmul.allow_tf32 = True
        torch.backends.cudnn.allow_tf32 = True
    for fn in ("enable_flash_sdp", "enable_mem_efficient_sdp"):
        if hasattr(torch.backends.cuda, fn):
            getattr(torch.backends.cuda, fn)(True)
print(f"GPU: {GPU_NAME} | VRAM {VRAM_GB:.0f} GB | bf16={USE_BF16}")

# Colab secrets → env (optional: Gemini query enhancement/VQA, HF pushes).
# GOOGLE_API_KEY is the name Colab's built-in "Gemini API key ▸ Import from
# Google AI Studio" button creates — the engine accepts either spelling.
try:
    from google.colab import userdata
    for _sec in ("GEMINI_API_KEY", "GOOGLE_API_KEY", "HF_TOKEN"):
        try:
            _v = userdata.get(_sec)
            if _v:
                os.environ[_sec] = _v
                print(f"secret {_sec}: loaded")
        except Exception:
            pass
except ImportError:
    pass

from cvp.config import load_settings
from cvp.utils.logging import setup_logging
settings = load_settings()
setup_logging("INFO")
print("data_root      =", settings.paths.data_root)
print("artifacts_root =", settings.paths.artifacts_root)

In [ ]:
# ── 5 · Load the search engine ──
# Pick the model for this session: "siglip2" | "finetuned" | "ensemble"
import os, time
os.environ["CVP_EMBEDDING__MODEL"] = "siglip2"     # ← change to test others
os.environ["CVP_QUERY__PROVIDER"]  = "none"        # offline test (no Gemini needed)

from cvp.search.engine import SearchEngine
from cvp.config import load_settings
settings = load_settings()
t0 = time.time()
engine = SearchEngine(settings)
print(f"engine ready in {time.time()-t0:.1f}s — {len(engine.catalog):,} keyframes")

In [ ]:
# ── 6 · Run sample Vietnamese KIS queries + show results inline ──
import time
import matplotlib.pyplot as plt
from PIL import Image

QUERIES = [
    "người dẫn chương trình mặc áo dài đứng trong trường quay",
    "đám cháy lớn khói đen bốc lên từ tòa nhà",
    "các vận động viên đua xe đạp trên đường phố",
    "món ăn được trình bày trên đĩa trắng",
    "cảnh ngập lụt trên đường phố, người dân lội nước",
]

for q in QUERIES:
    t0 = time.time()
    results = engine.search_text(q, display_k=8)
    dt = time.time() - t0
    print(f"\n🔎 {q}   ({dt:.2f}s)")
    if not results:
        print("   (no results)")
        continue
    fig, axes = plt.subplots(1, min(8, len(results)), figsize=(20, 3), squeeze=False)
    for ax, r in zip(axes[0], results):
        try:
            ax.imshow(Image.open(r.ref.path)); ax.axis("off")
            ax.set_title(f"{r.video_id}\nf={r.frame_idx} s={r.score:.2f}", fontsize=7)
        except Exception:
            ax.axis("off")
    plt.show()

In [ ]:
# ── 7 · TRAKE + AVS smoke test ──
events = [
    "vận động viên chuẩn bị xuất phát",
    "vận động viên chạy trên đường đua",
    "vận động viên về đích ăn mừng",
]
seqs = engine.search_trake(events, max_results=5)
for c in seqs[:5]:
    print(f"TRAKE {c.video_id} frames={c.frame_idxs} score={c.score:.3f}")

avs = engine.search_avs("cảnh giao thông đông đúc ở thành phố", limit=20)
print(f"\nAVS: {len(avs)} rows across {len({r.video_id for r in avs})} distinct videos")

In [ ]:
# ── 8 · Submission CSV round-trip (exact Codabench format) ──
from cvp.submission.writer import write_kis, write_trake

results = engine.search_text(QUERIES[0])
p = write_kis(settings.paths.art("submissions", "demo-kis.csv"),
              [(r.video_id, r.frame_idx) for r in results])
print(p, "\n" + p.read_text(encoding="utf-8")[:300])

nb03_files = [p]   # ONLY the CSVs this notebook run writes get packaged (cell 9)
if seqs:
    p2 = write_trake(settings.paths.art("submissions", "demo-trake.csv"),
                     [(c.video_id, c.frame_idxs) for c in seqs])
    nb03_files.append(p2)
    print(p2, "\n" + p2.read_text(encoding="utf-8")[:300])

In [ ]:
# ── 9 · Validate + package (Codabench) ──
# The organiser contract is re-checked on the finished CSVs (a wrong row
# silently costs a submission slot); errors BLOCK the zip. Validate and zip
# ONLY the CSVs cell 8 just wrote (nb03_files) — artifacts/submissions is
# shared with the UI's real exports, and stale/demo files must never mix
# into a contest zip (packager docstring contract).
import json
from cvp.submission.packager import has_errors, package_codabench, validate_file

sub_dir = settings.paths.art("submissions")
issues = [i for f in nb03_files for i in validate_file(f, strict=True)]
for i in issues:
    print(" ", i)
if has_errors(issues):
    print("❌ Còn lỗi chặn — sửa CSV rồi chạy lại ô này (zip KHÔNG được tạo).")
else:
    zip_path = settings.paths.art("submissions", "codabench.zip")
    package_codabench(sub_dir, zip_path, package_name=settings.submission.package_name,
                      files=nb03_files)
    manifest = json.loads((zip_path.parent / "MANIFEST.json").read_text(encoding="utf-8"))
    print(f"\n📦 {zip_path.name}  sha256={manifest['zip_sha256'][:12]}…")
    for f in manifest["files"]:
        print(f"   {f['name']:24} task={f['task']:5} rows={f['rows']}")

In [ ]:
# ── 10 · (optional) Score against ground truth — official formulas ──
# Drop a GT file at MyDrive/<project>/queries/gt.json to see the exact
# Codabench-style scores (R@k / Final) of the CSVs written above.
GT_PATH = PROJECT / "queries" / "gt.json"
if GT_PATH.exists():
    from cvp.eval.official import score_run
    report = score_run(settings.paths.art("submissions"), GT_PATH)
    print(f"{'query':28} {'task':6} {'final':>7} {'best_rank':>9}")
    for stem, qs in sorted(report.per_query.items()):
        print(f"{stem:28} {qs.task:6} {qs.final:>7.4f} {str(qs.best_rank):>9}")
    for stem, why in sorted(report.unscored.items()):
        print(f"{stem:28} UNSCORED: {why}")
    print(f"\nmean_final = {report.mean_final:.4f}  "
          f"({report.num_scored}/{report.num_gt} GT queries scored)")
else:
    print("Không thấy", GT_PATH, "— muốn chấm điểm offline, tạo JSON keyed theo")
    print("tên file CSV (không đuôi). Ví dụ tối thiểu:")
    print("""{
  "demo-kis": {"task": "kis", "video_id": "L21_V001", "range": [500, 510]}
}""")
    print("Hỗ trợ range / center+epsilon / moments / answers / targets (AVS coverage) "
          "— xem cvp/eval/official.py")

In [ ]:
# ── 11 · (optional) Automatic track dry-run (no human, no submit) ──
# Runs the full auto pipeline over MyDrive/<project>/queries/example/*.txt:
# infer task per filename → search → CSVs → validate → Codabench zip.
RUN_AUTO_AGENT = False
if RUN_AUTO_AGENT:
    from cvp.pipeline.auto_agent import run_auto
    qdir = PROJECT / "queries" / "example"
    qdir.mkdir(parents=True, exist_ok=True)
    if not any(qdir.glob("*.txt")):     # seed one sample query for the dry-run
        (qdir / "query-p1-1-kis.txt").write_text(
            "người dẫn chương trình mặc áo dài đứng trong trường quay", encoding="utf-8")
    # Reuse the cell-5 engine (review C6): the default engine_factory would
    # build a SECOND full engine and double index+catalog memory this session.
    rep = run_auto(qdir, settings.paths.art("submissions", "auto_dry_run"),
                   settings, submit=False, engine_factory=lambda _s: engine)
    print(f"AutoRunReport: written={len(rep.written)} ok={rep.ok} zip={rep.zip_path}")
    for i in rep.issues:
        print("  ", i)
else:
    print("RUN_AUTO_AGENT=False — bật để chạy thử automatic track (submit=False).")

In [ ]:
# ── 12 · (optional) Launch the Streamlit UI from Colab ──
# Colab can't open localhost — use the built-in proxy:
LAUNCH_UI = False
if LAUNCH_UI:
    import subprocess, time
    proc = subprocess.Popen(
        ["streamlit", "run", str(REPO_DIR / "app" / "streamlit_app.py"),
         "--server.port", "8501", "--server.headless", "true"])
    time.sleep(8)
    from google.colab import output
    output.serve_kernel_port_as_window(8501)
    # proc.terminate() when done
else:
    print("Set LAUNCH_UI=True to serve the app (better: run it on your laptop).")